# 07: Offline Evaluation

This notebook walks through the **Phase 2 offline evaluation suite** for the
Knowledge-Grounded QA Agent. Each evaluator runs *after* the agent has responded,
using a Langfuse dataset as the source of truth.

## What You'll Learn

0. **Browse questions** and add per-question ground truth annotations
1. How to run the agent and capture the full `AgentResponse` (plan, tool calls, sources)
2. **Replanning Rate** — deterministic counter (no LLM needed)
3. **Plan Quality** — LLM judge on the research plan
4. **Tool Selection & Efficiency** — LLM judge on the tool call sequence
5. **Source Validation** — LLM judge on source authority and relevance
6. **Knowledge Base Usage** — checks vertex_search was called when required
7. Running the full experiment against the ground truth dataset from Notebook 06

## Prerequisites

Complete Notebooks 01–06 (especially Notebook 06 — ground truth must be uploaded
to Langfuse before running the full experiment). All credentials in `.env`:
- `GOOGLE_API_KEY`
- `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY`
- `OPENAI_API_KEY` (for LLM judges)
- `VERTEX_AI_DATASTORE_ID` (optional — only for KB evaluation)

In [ ]:
import json
import os
from pathlib import Path
from typing import Any

import pandas as pd
from aieng.agent_evals.evaluation import run_experiment
from aieng.agent_evals.knowledge_qa import DeepSearchQADataset, KnowledgeGroundedAgent
from aieng.agent_evals.knowledge_qa.data.ground_truth import get_annotations
from aieng.agent_evals.knowledge_qa.evaluation.graders.plan_quality import (
    derive_plan_rubric,
    evaluate_plan_quality,
)
from aieng.agent_evals.knowledge_qa.evaluation.graders.replanning import (
    evaluate_replanning_rate,
    get_max_replan_threshold,
)
from aieng.agent_evals.knowledge_qa.evaluation.graders.source_validation import (
    evaluate_source_validation,
    get_source_rubric,
)
from aieng.agent_evals.knowledge_qa.evaluation.graders.tool_selection import (
    derive_tool_pattern,
    evaluate_tool_selection,
)
from aieng.agent_evals.knowledge_qa.evaluation.offline import (
    KnowledgeQATask,
    deepsearchqa_evaluator,
    knowledge_base_evaluator,
    plan_quality_evaluator,
    replanning_evaluator,
    source_validation_evaluator,
    tool_selection_evaluator,
)
from aieng.agent_evals.knowledge_qa.notebook import display_response, run_with_display
from dotenv import load_dotenv
from IPython.display import HTML, display  # noqa: A004
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


if Path("").absolute().name == "eval-agents":
    print(f"Working directory: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"Working directory set to: {Path('').absolute()}")

load_dotenv(verbose=True)
console = Console(width=100)

# Ground truth dataset uploaded in Notebook 06
DATASET_NAME = "KnowledgeQA-GroundTruth"
ANNOTATIONS_PATH = Path("implementations/knowledge_qa/data/ground_truth_annotations.jsonl")

## 0. Browse Questions and Add Ground Truth Annotations

The LLM judges (Plan Quality, Tool Selection) are more accurate when they have
per-question ground truth to anchor on — specifically:

| Field | Evaluator | Why it matters |
|---|---|---|
| `plan_rubric.must_cover` | Plan Quality | Tells the judge which concepts the plan must address |
| `tool_pattern.reference_tool_call_count` | Tool Selection | Tells the judge how many calls an expert would make |

Run the cell below to print all questions with their IDs, then fill in
`data/ground_truth_annotations.jsonl` for the ones you want to annotate.
All other fields are auto-derived — you only need to add what you know.

In [ ]:
# Load the dataset and print questions with their IDs
# Use this output to identify which example_ids to annotate
dataset = DeepSearchQADataset()

BROWSE_CATEGORY = "Finance & Economics"  # change to any category or set to None for all
BROWSE_LIMIT = 10

examples = (
    dataset.get_by_category(BROWSE_CATEGORY)[:BROWSE_LIMIT]
    if BROWSE_CATEGORY
    else dataset.examples[:BROWSE_LIMIT]
)

t = Table(title=f"Questions ({BROWSE_CATEGORY or 'All'}, first {BROWSE_LIMIT})")
t.add_column("example_id", style="yellow", justify="right", width=10)
t.add_column("answer_type", style="dim", width=14)
t.add_column("Question", style="white")
t.add_column("Answer (GT)", style="cyan", width=30)
for ex in examples:
    t.add_row(
        str(ex.example_id),
        ex.answer_type,
        ex.problem[:80] + "..." if len(ex.problem) > 80 else ex.problem,
        ex.answer[:28] + "..." if len(ex.answer) > 28 else ex.answer,
    )
console.print(t)

### How to add annotations

After identifying a question above, open `data/ground_truth_annotations.jsonl`
and add one line per question in this format:

```json
{"example_id": 1234, "plan_rubric": {"must_cover": ["GDP growth rate", "Statistics Canada source", "2023 timeframe"]}, "tool_pattern": {"reference_tool_call_count": 3}}
```

Only add the fields you want to override — everything else is auto-derived.
Then re-run the cell below to verify the annotation was loaded.

In [ ]:
annotations = get_annotations()
console.print(f"Annotations loaded: [bold]{len(annotations)}[/bold] items from {ANNOTATIONS_PATH}")

if len(annotations) == 0:
    console.print(
        "[dim]No annotations yet. Add entries to data/ground_truth_annotations.jsonl\n"
        "The evaluators will auto-derive rubrics for unannotated items.[/dim]"
    )
else:
    # Show which example_ids have annotations and what fields are covered
    ann_table = Table(title="Loaded Annotations")
    ann_table.add_column("example_id", style="yellow", justify="right")
    ann_table.add_column("plan_rubric.must_cover", style="cyan")
    ann_table.add_column("reference_tool_call_count", style="white", justify="right")
    for ex in examples:
        ann = annotations.get(ex.example_id)
        if ann:
            must_cover = ann.get("plan_rubric", {}).get("must_cover", [])
            ref_count = ann.get("tool_pattern", {}).get("reference_tool_call_count", "auto")
            ann_table.add_row(
                str(ex.example_id),
                ", ".join(must_cover) if must_cover else "(not set)",
                str(ref_count),
            )
    console.print(ann_table)

## 1. Run the Agent

All offline evaluators take their inputs from the `AgentResponse` object.
We run the agent once here and use the result across all subsequent sections.

In [ ]:
example = examples[0]

console.print(
    Panel(
        f"[bold]ID:[/bold] {example.example_id}\n"
        f"[bold]Category:[/bold] {example.problem_category}\n"
        f"[bold]Answer Type:[/bold] {example.answer_type}\n\n"
        f"[bold cyan]Question:[/bold cyan]\n{example.problem}\n\n"
        f"[bold yellow]Ground Truth:[/bold yellow]\n{example.answer}",
        title="Evaluation Example",
        border_style="blue",
    )
)

agent = KnowledgeGroundedAgent(enable_planning=True)
response = await run_with_display(agent, example.problem)

display_response(
    console,
    response.text,
    title="Agent Answer",
    subtitle=f"Duration: {response.total_duration_ms / 1000:.1f}s  |  Tools: {len(response.tool_calls)}  |  Replan count: {response.replan_count}",
)

## 2. Replanning Rate

Fully deterministic — no annotation needed. Reads `replan_count` directly from
`AgentResponse` and compares against the auto-derived threshold.

| Score | Meaning |
|---|---|
| `Replanning/Count` | Raw number of replannings |
| `Replanning/Flag` | 1 if count exceeds the threshold for this question type |
| `Replanning/Ratio` | replan_count / plan_steps |

In [ ]:
threshold = get_max_replan_threshold(
    answer_type=example.answer_type,
    problem_category=example.problem_category,
)
plan_steps = len(response.plan.steps) if response.plan else 1

replan_evals = evaluate_replanning_rate(
    replan_count=response.replan_count,
    plan_steps=plan_steps,
    max_replan_threshold=threshold,
)

t = Table(title="Replanning Rate")
t.add_column("Metric", style="cyan")
t.add_column("Value", style="white")
t.add_column("Note", style="dim")
for ev in replan_evals:
    note = f"threshold={threshold}" if ev.name == "Replanning/Flag" else ""
    t.add_row(ev.name, str(ev.value), note)
console.print(t)

## 3. Plan Quality

LLM judge on five dimensions. The rubric is auto-derived, but if you added
`must_cover` in the annotations file for this `example_id`, it will be merged
in automatically — giving the judge a concrete checklist to work from.

In [ ]:
plan_rubric = derive_plan_rubric(
    answer_type=example.answer_type,
    problem_category=example.problem_category,
)

# Merge annotation overrides if this example has them
annotation_overrides = annotations.get_plan_rubric_overrides(example.example_id)
if annotation_overrides:
    plan_rubric = {**plan_rubric, **annotation_overrides}
    console.print(f"[green]✓[/green] Applied annotation overrides for example_id={example.example_id}")
else:
    console.print("[dim]No annotation for this example — using auto-derived rubric[/dim]")
console.print("[dim]Plan rubric:[/dim]", plan_rubric)

plan_descriptions = [
    step.description for step in response.plan.steps
] if response.plan else []

plan_evals = await evaluate_plan_quality(
    question=example.problem,
    plan_steps=plan_descriptions,
    plan_rubric=plan_rubric,
)

t = Table(title="Plan Quality")
t.add_column("Dimension", style="cyan")
t.add_column("Score", style="white", justify="right")
t.add_column("Comment", style="dim")
for ev in plan_evals:
    t.add_row(ev.name, str(int(ev.value)), ev.comment or "")
console.print(t)

## 4. Tool Selection & Efficiency

If you annotated `reference_tool_call_count` for this question, the judge uses
that instead of the category average for scoring `Efficiency/CallVolume`.

In [ ]:
tool_pattern = derive_tool_pattern(
    problem_category=example.problem_category,
    answer_type=example.answer_type,
)

# Merge annotation overrides
tool_overrides = annotations.get_tool_pattern_overrides(example.example_id)
if tool_overrides:
    tool_pattern = {**tool_pattern, **tool_overrides}
    console.print(f"[green]✓[/green] Applied tool annotation overrides for example_id={example.example_id}")
else:
    console.print("[dim]No annotation — using auto-derived tool pattern[/dim]")
console.print("[dim]Tool pattern:[/dim]", tool_pattern)

seq_table = Table(title=f"Tool Calls ({len(response.tool_calls)} total)")
seq_table.add_column("#", style="dim", justify="right")
seq_table.add_column("Tool", style="cyan")
seq_table.add_column("Args (preview)", style="white")
for i, tc in enumerate(response.tool_calls, 1):
    args_str = str(tc.get("args", {}))
    seq_table.add_row(str(i), tc.get("name", "?"), args_str[:60])
console.print(seq_table)

tool_evals = await evaluate_tool_selection(
    question=example.problem,
    tool_calls=response.tool_calls,
    final_answer=response.text,
    tool_pattern=tool_pattern,
)

t = Table(title="Tool Selection & Efficiency")
t.add_column("Dimension", style="cyan")
t.add_column("Score", style="white", justify="right")
t.add_column("Comment", style="dim")
for ev in tool_evals:
    t.add_row(ev.name, str(int(ev.value)), ev.comment or "")
console.print(t)

## 5. Source Validation

Fully auto-derived from `CATEGORY_SOURCE_RUBRIC` — no annotation needed.
vertex_search sources are skipped (already grounded in the private KB).

In [ ]:
source_rubric = get_source_rubric(example.problem_category)
console.print("[dim]Source rubric:[/dim]", source_rubric)

sources_dict = [{"title": s.title, "uri": s.uri} for s in response.sources]
console.print(f"[dim]{len(sources_dict)} sources cited[/dim]")

source_evals = await evaluate_source_validation(
    question=example.problem,
    problem_category=example.problem_category,
    sources=sources_dict,
    source_rubric=source_rubric,
)

t = Table(title="Source Validation")
t.add_column("Dimension", style="cyan")
t.add_column("Score", style="white", justify="right")
t.add_column("Comment", style="dim")
for ev in source_evals:
    t.add_row(ev.name, str(int(ev.value)), ev.comment or "")
console.print(t)

## 6. Full Experiment: All Evaluators via `run_experiment`

Runs against the `KnowledgeQA-GroundTruth` dataset uploaded in Notebook 06.
Each item already has `must_cover`, `reference_tool_call_count`, `source_rubric`,
and `requires_knowledge_base` baked into its metadata — the judges receive
concrete ground truth instead of auto-derived defaults.

> One agent call + six evaluator calls per dataset item.
> With 9 items and `max_concurrency=2`, expect ~15–30 minutes.

In [ ]:
task = KnowledgeQATask()

experiment_result = run_experiment(
    DATASET_NAME,
    name="knowledge-agent-full-offline-eval",
    description="All Phase 2 offline evaluators: F1, replanning, plan quality, tool selection, source validation",
    task=task.run,
    evaluators=[
        deepsearchqa_evaluator,
        replanning_evaluator,
        plan_quality_evaluator,
        tool_selection_evaluator,
        knowledge_base_evaluator,
        source_validation_evaluator,
    ],
    max_concurrency=2,
)

console.print("[green]Experiment complete[/green]")
if hasattr(experiment_result, "dataset_run_url") and experiment_result.dataset_run_url:
    display(HTML(
        f'<p>View in Langfuse: <a href="{experiment_result.dataset_run_url}" target="_blank">'
        f'{experiment_result.dataset_run_url}</a></p>'
    ))

## 7. Inspecting Results

In [ ]:
rows = []
for item_result in experiment_result.item_results:
    item = item_result.item
    question = str(item.input)
    row = {"question": question[:50] + "..." if len(question) > 50 else question}
    for ev in item_result.evaluations or []:
        row[ev.name] = ev.value
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

numeric_metrics = [c for c in df.columns if c != "question"]
if numeric_metrics:
    means = Table(title="Mean Scores Across Dataset")
    means.add_column("Metric", style="cyan")
    means.add_column("Mean", style="white", justify="right")
    for col in sorted(numeric_metrics):
        if df[col].dtype in ("float64", "int64"):
            means.add_row(col, f"{df[col].mean():.3f}")
    console.print(means)

## Summary

In this notebook you:

0. **Browsed questions** by category, found their `example_id` values, and added
   `must_cover` / `reference_tool_call_count` annotations to `ground_truth_annotations.jsonl`
1. **Ran** the agent and captured the full `AgentResponse`
2. **Evaluated replanning rate** deterministically (no annotation needed)
3. **Evaluated plan quality** with annotations merged in for `must_cover`
4. **Evaluated tool selection** with `reference_tool_call_count` from annotations
5. **Evaluated source quality** fully auto-derived
6. **Ran the full experiment** with all evaluators — annotations applied automatically
7. **Inspected results** programmatically and in Langfuse

### Annotation workflow going forward

1. Run Section 0 to browse questions and note their `example_id` values
2. Add entries to `data/ground_truth_annotations.jsonl`
3. Re-run `python data/langfuse_upload.py` — annotations are merged into the Langfuse metadata
4. Re-run the experiment — annotated items get richer rubrics automatically